# System Overview — Trier Study Area

Visualises the **current state of the system**: the study area boundary, the road network,
all Points of Interest, the functional infrastructure (power grid & drinking water),
the **HQ100 flood-hazard area**, and the **RoA reference-point grid**.

| Layer | Description |
|---|---|
| City boundary | Administrative boundary of Trier, Germany |
| Road network | `drive_service` graph from OpenStreetMap |
| **Flood hazard area** | **HQ100 inundation polygon (100-year return period)** |
| **RoA sample grid** | **Reference nodes from which accessibility is measured (~500 m spacing shown)** |
| Hospitals | Emergency-care facilities |
| Fire stations | Fire-fighting stations |
| Rivers / waterways | River bodies and centrelines |
| Power substations | Electrical substations & switching stations |
| Power lines | High-voltage transmission lines |
| Power plants | Generation facilities |
| Water towers | Elevated storage / distribution nodes |
| Water works | Treatment / source facilities |
| Pumping stations | Pumping stations in the water network |

**Output:** static Matplotlib overview + interactive Folium map (`data/output/system_overview_map.html`).

> **Note on the sample grid:** The full RoA analysis (notebook 03) uses a 50 m spacing,
> yielding thousands of reference nodes. Here we draw a coarser 500 m grid so the map
> stays readable. The sampling logic is identical.

In [ ]:
from __future__ import annotations

import osmnx as ox
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from shapely.ops import unary_union
from pathlib import Path

ox.settings.log_console = False
ox.settings.use_cache = True

print(f"osmnx    : {ox.__version__}")
print(f"folium   : {folium.__version__}")
print(f"geopandas: {gpd.__version__}")

## 1. Configuration & Paths

In [ ]:
from css_geodata_service.robustness_of_accessibility.examples.notebooks.notebook_utils import (
    RoaNotebookConfig,
    get_roa_cache_path,
    get_roa_hazard_data_path,
    get_roa_outputs_path,
    set_notbook_wd,
)

set_notbook_wd()

place_name: str = RoaNotebookConfig.place_name      # "Trier, Germany"
network_type: str = RoaNotebookConfig.network_type  # "drive_service"
event = RoaNotebookConfig.event                     # HQ100
cache_dir: Path = get_roa_cache_path()
output_dir: Path = get_roa_outputs_path()
hazard_data_path: Path = get_roa_hazard_data_path(event=event)

output_dir.mkdir(parents=True, exist_ok=True)
(cache_dir / "network").mkdir(parents=True, exist_ok=True)
(cache_dir / "services").mkdir(parents=True, exist_ok=True)

print(f"Place          : {place_name}")
print(f"Network type   : {network_type}")
print(f"Hazard event   : {event}")
print(f"Hazard data    : {hazard_data_path}")
print(f"Cache dir      : {cache_dir}")
print(f"Output dir     : {output_dir}")

## 2. Study-Area Boundary

In [ ]:
place_gdf = ox.geocode_to_gdf(place_name)

if len(place_gdf) > 1:
    raise RuntimeError(f"Multiple OSM entries found for '{place_name}'")

boundary_geom = place_gdf.geometry.iloc[0]

if boundary_geom.geom_type == "MultiPolygon":
    boundary_geom = unary_union(boundary_geom)
elif boundary_geom.geom_type != "Polygon":
    raise RuntimeError(f"Unexpected geometry type: {boundary_geom.geom_type}")

boundary_gdf = gpd.GeoDataFrame(geometry=[boundary_geom], crs="EPSG:4326")

minx, miny, maxx, maxy = boundary_geom.bounds
center_lat = (miny + maxy) / 2
center_lon = (minx + maxx) / 2

print(f"Bounds  : lon [{minx:.4f}, {maxx:.4f}]  lat [{miny:.4f}, {maxy:.4f}]")
print(f"Centre  : ({center_lat:.4f}, {center_lon:.4f})")
boundary_geom

## 3. Road Network

In [ ]:
network_cache = cache_dir / f"network/drive_graph_{place_name}.graphml"

if network_cache.exists():
    print("Loading road network from cache ...")
    road_network = ox.load_graphml(network_cache)
else:
    print("Downloading road network from OSM (may take ~1–2 min) ...")
    road_network = ox.graph_from_polygon(polygon=boundary_geom, network_type=network_type)
    ox.save_graphml(road_network, filepath=network_cache)
    print(f"  → Saved to cache: {network_cache}")

road_network_nodes, road_network_edges = ox.graph_to_gdfs(road_network)
print(f"Nodes : {len(road_network_nodes):,}")
print(f"Edges : {len(road_network_edges):,}")

## 4. Emergency-Service POIs & Water Features

In [ ]:
def _load_or_fetch(cache_path: Path, polygon, tags: dict) -> gpd.GeoDataFrame:
    """Load a GeoDataFrame from cache or download via osmnx."""
    if cache_path.exists():
        return gpd.read_file(cache_path)
    gdf = ox.features_from_polygon(polygon=polygon, tags=tags)
    gdf.to_file(cache_path, driver="GeoJSON")
    return gdf


hospitals = _load_or_fetch(
    cache_dir / f"services/hospitals_{place_name}.geojson",
    boundary_geom, {"amenity": ["hospital"]}
)
fire_stations = _load_or_fetch(
    cache_dir / f"services/fire_stations_{place_name}.geojson",
    boundary_geom, {"amenity": ["fire_station"]}
)

rivers = ox.features_from_polygon(polygon=boundary_geom, tags={"water": ["river"]})
waterways = ox.features_from_polygon(polygon=boundary_geom, tags={"waterway": ["river"]})

print(f"Hospitals     : {len(hospitals)}")
print(f"Fire stations : {len(fire_stations)}")
print(f"River polygons: {len(rivers)}")
print(f"River lines   : {len(waterways)}")

## 5. Power Infrastructure

Uses OSM `power=*` tags (paper section 2.1). Verifiable at [Open Infrastructure Map](https://openinframap.org/).

In [ ]:
power_substations = _load_or_fetch(
    cache_dir / f"services/power_substations_{place_name}.geojson",
    boundary_geom, {"power": ["substation"]}
)
power_plants = _load_or_fetch(
    cache_dir / f"services/power_plants_{place_name}.geojson",
    boundary_geom, {"power": ["plant"]}
)
power_lines = _load_or_fetch(
    cache_dir / f"services/power_lines_{place_name}.geojson",
    boundary_geom, {"power": ["line", "minor_line"]}
)

power_line_geoms = power_lines[
    power_lines.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()
hv_lines = power_line_geoms[
    power_line_geoms.get("power", pd.Series(dtype=str)) == "line"
] if "power" in power_line_geoms.columns else power_line_geoms

print(f"Power substations : {len(power_substations)}")
print(f"Power plants      : {len(power_plants)}")
print(f"HV lines          : {len(hv_lines)}")

## 6. Drinking-Water Infrastructure

In [ ]:
water_towers = _load_or_fetch(
    cache_dir / f"services/water_towers_{place_name}.geojson",
    boundary_geom, {"man_made": ["water_tower"]}
)
water_works = _load_or_fetch(
    cache_dir / f"services/water_works_{place_name}.geojson",
    boundary_geom, {"man_made": ["water_works"]}
)
pumping_stations = _load_or_fetch(
    cache_dir / f"services/pumping_stations_{place_name}.geojson",
    boundary_geom, {"man_made": ["pumping_station"]}
)

print(f"Water towers      : {len(water_towers)}")
print(f"Water works       : {len(water_works)}")
print(f"Pumping stations  : {len(pumping_stations)}")

## 7. Flood Hazard Area (HQ100)

The flood polygon defines the road segments that are **removed** from the network
when computing the disrupted RoA score. The data covers the 100-year return-period
flood event (`HQ100`, event code `M`).

> If the hazard file is missing, download the data and place it under
> `data/input/Flooding/HazardAreas/` as described in the notebook prerequisites.

In [ ]:
hazard_area_gdf = None
hazard_area_multipolygon = None

if hazard_data_path.exists():
    hazard_area_gdf = gpd.read_file(hazard_data_path)
    hazard_area_multipolygon = unary_union(hazard_area_gdf.geometry)
    hazard_area_gs = gpd.GeoSeries([hazard_area_multipolygon], crs="EPSG:4326")
    print(f"Flood polygons loaded : {len(hazard_area_gdf)}")
    print(f"Combined geometry type: {hazard_area_multipolygon.geom_type}")
    print(f"Flood area bounds     : {hazard_area_multipolygon.bounds}")
else:
    print(f"WARNING: Hazard data file not found at:\n  {hazard_data_path}")
    print("Flood layer will be skipped in the maps.")

## 8. RoA Sample Grid (Reference Points)

The RoA score is computed **from every node in this grid** to the nearest POI,
in both the normal and the disrupted (flooded) network.  
There is no single reference point — the grid covers the whole city.

| Parameter | Value used here | Value in full analysis (notebook 03) |
|---|---|---|
| Grid spacing | 500 m | 50 m |
| Node selection | 1 random road node per cell | 1 random road node per cell |
| Random seed | 42 | 42 |

In [ ]:
from css_geodata_service.robustness_of_accessibility.robustness_of_accessibility import draw_sample

SAMPLE_DISTANCE_VIZ = 500  # metres — coarser grid, easier to read

sample_points = draw_sample(
    polygon=boundary_geom,
    gdf_nodes_drive_service_graph=road_network_nodes,
    sample_distance_in_meters=SAMPLE_DISTANCE_VIZ,
    random_seed=42,
)
sample_gdf = gpd.GeoDataFrame(geometry=sample_points.geometry, crs="EPSG:4326")

print(f"Sample points at {SAMPLE_DISTANCE_VIZ} m spacing : {len(sample_gdf)}")
print(f"(Full analysis at 50 m would yield ~{len(sample_gdf) * 100:,}+ points)")

## 9. Summary

In [ ]:
print("=" * 55)
print(f"  TRIER STUDY AREA — SYSTEM OVERVIEW")
print("=" * 55)
print(f"  Area              : {place_name}")
print(f"  Bounds (lon/lat)  : [{minx:.4f},{maxx:.4f}] / [{miny:.4f},{maxy:.4f}]")
print()
print(f"  Road Network")
print(f"    Nodes           : {len(road_network_nodes):,}")
print(f"    Edges           : {len(road_network_edges):,}")
print()
print(f"  Flood Hazard (HQ100)")
flood_info = f"{len(hazard_area_gdf)} source polygon(s)" if hazard_area_gdf is not None else "NOT LOADED"
print(f"    Data            : {flood_info}")
print()
print(f"  RoA Sample Grid")
print(f"    Points (500m)   : {len(sample_gdf)}")
print()
print(f"  Emergency Services")
print(f"    Hospitals       : {len(hospitals)}")
print(f"    Fire stations   : {len(fire_stations)}")
print()
print(f"  Power Infrastructure")
print(f"    Substations     : {len(power_substations)}")
print(f"    Plants          : {len(power_plants)}")
print(f"    HV lines        : {len(hv_lines)}")
print()
print(f"  Drinking-Water Infrastructure")
print(f"    Water towers    : {len(water_towers)}")
print(f"    Water works     : {len(water_works)}")
print(f"    Pumping stations: {len(pumping_stations)}")
print("=" * 55)

## 10. Static Map — Matplotlib

In [ ]:
def to_point_gdf(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = gdf.copy()
    out["geometry"] = out["geometry"].centroid
    return out[out["geometry"].notna()]


hospitals_pts     = to_point_gdf(hospitals)
fire_stations_pts = to_point_gdf(fire_stations)
substations_pts   = to_point_gdf(power_substations)
plants_pts        = to_point_gdf(power_plants)
water_towers_pts  = to_point_gdf(water_towers)
water_works_pts   = to_point_gdf(water_works)
pumping_pts       = to_point_gdf(pumping_stations)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 12), facecolor="white")

# City boundary
boundary_gdf.plot(ax=ax, facecolor="#E8F4F8", edgecolor="#333333", linewidth=2.0, zorder=1)

# Flood hazard area
if hazard_area_multipolygon is not None:
    gpd.GeoSeries([hazard_area_multipolygon], crs="EPSG:4326").plot(
        ax=ax, color="#336699", alpha=0.45, zorder=2
    )

# Road network
road_network_edges.plot(ax=ax, color="#BBBBBB", linewidth=0.35, alpha=0.8, zorder=3)

# RoA sample grid
sample_gdf.plot(
    ax=ax, color="#444444", markersize=14, marker="+",
    zorder=4, alpha=0.75, linewidths=0.8,
)

# Rivers
if not rivers.empty:
    rivers.plot(ax=ax, color="#4488CC", alpha=0.45, zorder=3)
if not waterways.empty:
    ww_lines = waterways[waterways.geometry.geom_type.isin(["LineString", "MultiLineString"])]
    if not ww_lines.empty:
        ww_lines.plot(ax=ax, color="#2266BB", linewidth=1.5, alpha=0.8, zorder=3)

# Power lines
if not hv_lines.empty:
    hv_lines.plot(ax=ax, color="#FF8C00", linewidth=1.0, alpha=0.75, zorder=5)

# Power substations & plants
if not substations_pts.empty:
    substations_pts.plot(ax=ax, color="#FF6600", markersize=55, marker="s",
                         zorder=7, edgecolors="white", linewidths=0.8)
if not plants_pts.empty:
    plants_pts.plot(ax=ax, color="#CC4400", markersize=100, marker="D",
                    zorder=7, edgecolors="white", linewidths=0.8)

# Water infrastructure
if not water_works_pts.empty:
    water_works_pts.plot(ax=ax, color="#007ACC", markersize=80, marker="s",
                         zorder=7, edgecolors="white", linewidths=0.8)
if not water_towers_pts.empty:
    water_towers_pts.plot(ax=ax, color="#00BFFF", markersize=60, marker="o",
                          zorder=7, edgecolors="white", linewidths=0.8)
if not pumping_pts.empty:
    pumping_pts.plot(ax=ax, color="#009999", markersize=50, marker="v",
                     zorder=7, edgecolors="white", linewidths=0.8)

# Emergency POIs
if not hospitals_pts.empty:
    hospitals_pts.plot(ax=ax, color="#CC2222", markersize=90, marker="o",
                       zorder=8, edgecolors="white", linewidths=1.0)
if not fire_stations_pts.empty:
    fire_stations_pts.plot(ax=ax, color="#1144CC", markersize=70, marker="^",
                           zorder=8, edgecolors="white", linewidths=1.0)

# ── Legend ─────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor="#E8F4F8", edgecolor="#333333", linewidth=1.5, label="City Boundary"),
    mpatches.Patch(facecolor="#336699", alpha=0.6, label="Flood Hazard — HQ100"),
    mpatches.Patch(facecolor="#BBBBBB", label=f"Road Network ({len(road_network_edges):,} edges)"),
    plt.Line2D([0],[0], marker="+", color="#444444", markersize=10, linewidth=0,
               label=f"RoA Sample Grid — {SAMPLE_DISTANCE_VIZ} m ({len(sample_gdf)} pts)"),
    mpatches.Patch(facecolor="#4488CC", alpha=0.7, label="Rivers / Waterways"),
    # Power
    plt.Line2D([0],[0], color="#FF8C00", linewidth=2, label=f"HV Power Lines ({len(hv_lines)})"),
    plt.Line2D([0],[0], marker="s", color="w", markerfacecolor="#FF6600",
               markersize=10, label=f"Power Substations ({len(substations_pts)})"),
    plt.Line2D([0],[0], marker="D", color="w", markerfacecolor="#CC4400",
               markersize=10, label=f"Power Plants ({len(plants_pts)})"),
    # Water
    plt.Line2D([0],[0], marker="s", color="w", markerfacecolor="#007ACC",
               markersize=10, label=f"Water Works ({len(water_works_pts)})"),
    plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="#00BFFF",
               markersize=10, label=f"Water Towers ({len(water_towers_pts)})"),
    plt.Line2D([0],[0], marker="v", color="w", markerfacecolor="#009999",
               markersize=10, label=f"Pumping Stations ({len(pumping_pts)})"),
    # Emergency
    plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="#CC2222",
               markersize=12, label=f"Hospitals ({len(hospitals_pts)})"),
    plt.Line2D([0],[0], marker="^", color="w", markerfacecolor="#1144CC",
               markersize=12, label=f"Fire Stations ({len(fire_stations_pts)})"),
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=9, framealpha=0.93)

ax.set_title(f"{place_name} — System Overview", fontsize=15, fontweight="bold", pad=16)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(output_dir / "system_overview_static.png", dpi=200, bbox_inches="tight")
plt.show()
print("Static map saved.")

## 11. Interactive Map — Folium

Toggle layers with the **layer control** (top-right). Click markers for names.

In [ ]:
def _get_name(row: pd.Series, fallback: str = "–") -> str:
    val = row.get("name") if "name" in row.index else None
    return str(val) if val is not None and pd.notna(val) else fallback


# ── Base map ──────────────────────────────────────────────────────────────────
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles="CartoDB positron",
)

# ── City boundary ─────────────────────────────────────────────────────────────
boundary_layer = folium.FeatureGroup(name="City Boundary", show=True)
folium.GeoJson(
    boundary_gdf.__geo_interface__,
    style_function=lambda _: {
        "fillColor": "#B0D4E8", "fillOpacity": 0.18,
        "color": "#333333", "weight": 2.5,
    },
    tooltip="Trier, Germany — Study Area Boundary",
).add_to(boundary_layer)
boundary_layer.add_to(m)

# ── Flood hazard area ─────────────────────────────────────────────────────────
if hazard_area_multipolygon is not None:
    flood_layer = folium.FeatureGroup(name="Flood Hazard — HQ100", show=True)
    geo_json_flooded = folium.GeoJson(
        gpd.GeoSeries([hazard_area_multipolygon], crs="EPSG:4326").__geo_interface__,
        style_function=lambda _: {
            "fillColor": "#336699",
            "fillOpacity": 0.50,
            "color": "rgba(0,0,0,0)",
        },
        tooltip="Flood Hazard Area — HQ100 (100-year return period)",
    )
    folium.Popup("Flood scenario: 100-year return period (HQ100)").add_to(geo_json_flooded)
    geo_json_flooded.add_to(flood_layer)
    flood_layer.add_to(m)

# ── Road network ──────────────────────────────────────────────────────────────
road_layer = folium.FeatureGroup(name="Road Network", show=True)
edges_for_folium = road_network_edges[["geometry"]].reset_index(drop=True)
folium.GeoJson(
    edges_for_folium.__geo_interface__,
    style_function=lambda _: {"color": "#999999", "weight": 0.8, "opacity": 0.6},
).add_to(road_layer)
road_layer.add_to(m)

# ── RoA sample grid ───────────────────────────────────────────────────────────
sample_layer = folium.FeatureGroup(
    name=f"RoA Sample Grid — {SAMPLE_DISTANCE_VIZ} m ({len(sample_gdf)} pts)",
    show=True,
)
for _, row in sample_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="#444444",
        weight=1.5,
        fill=True,
        fill_color="#666666",
        fill_opacity=0.70,
        tooltip="RoA reference node",
    ).add_to(sample_layer)
sample_layer.add_to(m)

# ── Rivers & waterways ────────────────────────────────────────────────────────
if not rivers.empty or not waterways.empty:
    river_layer = folium.FeatureGroup(name="Rivers & Waterways", show=True)
    if not rivers.empty:
        folium.GeoJson(
            rivers[["geometry"]].reset_index(drop=True).__geo_interface__,
            style_function=lambda _: {
                "fillColor": "#4488CC", "fillOpacity": 0.40,
                "color": "#2266AA", "weight": 1.0,
            },
        ).add_to(river_layer)
    if not waterways.empty:
        ww_ln = waterways[waterways.geometry.geom_type.isin(["LineString", "MultiLineString"])]
        if not ww_ln.empty:
            folium.GeoJson(
                ww_ln[["geometry"]].reset_index(drop=True).__geo_interface__,
                style_function=lambda _: {"color": "#2266AA", "weight": 2.5, "opacity": 0.85},
            ).add_to(river_layer)
    river_layer.add_to(m)

# ── Power lines ───────────────────────────────────────────────────────────────
if not hv_lines.empty:
    power_line_layer = folium.FeatureGroup(name=f"Power Lines — HV ({len(hv_lines)})", show=True)
    folium.GeoJson(
        hv_lines[["geometry"]].reset_index(drop=True).__geo_interface__,
        style_function=lambda _: {"color": "#FF8C00", "weight": 1.8, "opacity": 0.80},
        tooltip="High-voltage transmission line",
    ).add_to(power_line_layer)
    power_line_layer.add_to(m)

# ── Power substations & plants ────────────────────────────────────────────────
if not substations_pts.empty or not plants_pts.empty:
    substation_layer = folium.FeatureGroup(
        name=f"Power Substations ({len(substations_pts)}) & Plants ({len(plants_pts)})",
        show=True,
    )
    for _, row in substations_pts.iterrows():
        label = _get_name(row, "Substation")
        voltage = str(row.get("voltage", "–")) if "voltage" in row.index else "–"
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=8, color="#CC5500", weight=1.5,
            fill=True, fill_color="#FF6600", fill_opacity=0.88,
            popup=folium.Popup(f"<b>Power Substation</b><br>{label}<br>Voltage: {voltage} V", max_width=260),
            tooltip=f"Substation: {label}",
        ).add_to(substation_layer)
    for _, row in plants_pts.iterrows():
        label = _get_name(row, "Power Plant")
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=12, color="#993300", weight=2,
            fill=True, fill_color="#CC4400", fill_opacity=0.90,
            popup=folium.Popup(f"<b>Power Plant</b><br>{label}", max_width=260),
            tooltip=f"Plant: {label}",
        ).add_to(substation_layer)
    substation_layer.add_to(m)

# ── Water infrastructure ──────────────────────────────────────────────────────
n_water = len(water_towers_pts) + len(water_works_pts) + len(pumping_pts)
water_infra_layer = folium.FeatureGroup(name=f"Drinking-Water Infrastructure ({n_water})", show=True)

for _, row in water_works_pts.iterrows():
    label = _get_name(row, "Water Works")
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=11, color="#005599", weight=2,
        fill=True, fill_color="#007ACC", fill_opacity=0.90,
        popup=folium.Popup(f"<b>Water Works</b><br>{label}", max_width=260),
        tooltip=f"Water Works: {label}",
    ).add_to(water_infra_layer)

for _, row in water_towers_pts.iterrows():
    label = _get_name(row, "Water Tower")
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=9, color="#006699", weight=1.5,
        fill=True, fill_color="#00BFFF", fill_opacity=0.88,
        popup=folium.Popup(f"<b>Water Tower</b><br>{label}", max_width=260),
        tooltip=f"Water Tower: {label}",
    ).add_to(water_infra_layer)

for _, row in pumping_pts.iterrows():
    label = _get_name(row, "Pumping Station")
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=7, color="#007777", weight=1.5,
        fill=True, fill_color="#009999", fill_opacity=0.88,
        popup=folium.Popup(f"<b>Pumping Station</b><br>{label}", max_width=260),
        tooltip=f"Pumping Station: {label}",
    ).add_to(water_infra_layer)

water_infra_layer.add_to(m)

# ── Hospitals ─────────────────────────────────────────────────────────────────
hospital_layer = folium.FeatureGroup(name=f"Hospitals ({len(hospitals_pts)})", show=True)
for _, row in hospitals_pts.iterrows():
    label = _get_name(row, "Hospital")
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=10, color="#991111", weight=2,
        fill=True, fill_color="#CC2222", fill_opacity=0.88,
        popup=folium.Popup(f"<b>Hospital</b><br>{label}", max_width=260),
        tooltip=label,
    ).add_to(hospital_layer)
hospital_layer.add_to(m)

# ── Fire stations ─────────────────────────────────────────────────────────────
fire_layer = folium.FeatureGroup(name=f"Fire Stations ({len(fire_stations_pts)})", show=True)
for _, row in fire_stations_pts.iterrows():
    label = _get_name(row, "Fire Station")
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=8, color="#112299", weight=2,
        fill=True, fill_color="#1144CC", fill_opacity=0.90,
        popup=folium.Popup(f"<b>Fire Station</b><br>{label}", max_width=260),
        tooltip=label,
    ).add_to(fire_layer)
fire_layer.add_to(m)

# ── Layer control & save ──────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(m)

map_path = output_dir / "system_overview_map.html"
m.save(str(map_path))
print(f"Interactive map saved: {map_path}")

m

## 12. NCNN Routes — POI to Infrastructure

For each emergency-service POI the **Network-Constrained Nearest-Neighbor (NCNN)**
algorithm finds the nearest power / water infrastructure node using the
**road-network distance** (not straight-line distance).

This reflects the physical reality stated in the paper: power cables and water
pipes run underground *along street corridors*, so road-network distance is the
appropriate proximity metric — even though the infrastructure itself is not on
the road surface.

**Algorithm** (reuses the existing multi-source Dijkstra):
1. Snap every infrastructure centroid to its nearest road node.
2. Run `nx.multi_source_dijkstra` from *all* infrastructure nodes simultaneously.
3. For each POI road-node, read off the distance and path to the nearest source.
4. Reconstruct the `MultiLineString` route geometry.
5. Persist results as GeoJSON in `data/processed/ncnn/` (cached for later notebooks).

| Layer | Description |
|---|---|
| Hospital → Power | Nearest power substation/plant via road network |
| Hospital → Water | Nearest water works/tower/pumping station via road network |
| Fire Station → Power | Nearest power substation/plant via road network |
| Fire Station → Water | Nearest water works/tower/pumping station via road network |

In [ ]:
import pandas as pd
from css_geodata_service.robustness_of_accessibility.utils.ncnn import load_or_calculate_ncnn_routes

# Use an undirected graph so one-way restrictions do not block underground
# infrastructure paths (power/water pipes are bidirectional by nature).
road_network_undirected = road_network.to_undirected()

# ── Combine infrastructure types ───────────────────────────────────────────────
# Water: water works + water towers + pumping stations → one GeoDataFrame
water_infra_combined = gpd.GeoDataFrame(
    pd.concat([water_works, water_towers, pumping_stations], ignore_index=True),
    crs="EPSG:4326",
)

# Power: substations + plants → one GeoDataFrame
power_infra_combined = gpd.GeoDataFrame(
    pd.concat([power_substations, power_plants], ignore_index=True),
    crs="EPSG:4326",
)

poi_gdfs = {
    "hospital":     hospitals,
    "fire_station": fire_stations,
}
infra_gdfs = {
    "power": power_infra_combined,
    "water": water_infra_combined,
}

# ── Compute (or load from cache) ───────────────────────────────────────────────
print("Computing / loading NCNN routes (POI → nearest infrastructure via road network) …")
ncnn_results = load_or_calculate_ncnn_routes(
    cache_dir=cache_dir,
    poi_gdfs=poi_gdfs,
    infrastructure_gdfs=infra_gdfs,
    street_network=road_network_undirected,
    place_name=place_name,
)
print("Done.\n")

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"{'POI type':15s}  {'Infra type':10s}  {'POIs':>4s}  {'Mean (m)':>9s}  {'Max (m)':>8s}")
print("-" * 57)
for poi_type, infra_dict in ncnn_results.items():
    for infra_type, gdf in infra_dict.items():
        finite = gdf[gdf["route_length_m"] < float("inf")]["route_length_m"]
        mean_m = f"{finite.mean():.0f}" if len(finite) else "—"
        max_m  = f"{finite.max():.0f}"  if len(finite) else "—"
        print(f"{poi_type:15s}  {infra_type:10s}  {len(gdf):>4d}  {mean_m:>9s}  {max_m:>8s}")

In [ ]:
# ── Interactive Folium map — NCNN routes ───────────────────────────────────────
# Route colours per (poi_type, infra_type)
_NCNN_STYLE = {
    ("hospital",     "power"): {"color": "#FF4500", "weight": 3.5, "dashArray": None},
    ("hospital",     "water"): {"color": "#1E90FF", "weight": 3.5, "dashArray": None},
    ("fire_station", "power"): {"color": "#FF8C00", "weight": 3.0, "dashArray": "6 4"},
    ("fire_station", "water"): {"color": "#00BFFF", "weight": 3.0, "dashArray": "6 4"},
}
_POI_COLOR   = {"hospital": "#CC2222", "fire_station": "#1144CC"}
_INFRA_COLOR = {"power": "#FF6600",    "water": "#007ACC"}
_POI_LABEL   = {"hospital": "Hospital", "fire_station": "Fire Station"}
_INFRA_LABEL = {"power": "Power Infrastructure", "water": "Water Infrastructure"}

ncnn_map = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles="CartoDB positron")

# Light road-network background
road_bg = folium.FeatureGroup(name="Road Network (background)", show=True)
edges_bg = road_network_edges[["geometry"]].reset_index(drop=True)
folium.GeoJson(
    edges_bg.__geo_interface__,
    style_function=lambda _: {"color": "#BBBBBB", "weight": 0.6, "opacity": 0.5},
).add_to(road_bg)
road_bg.add_to(ncnn_map)

# NCNN route layers
for poi_type, infra_dict in ncnn_results.items():
    for infra_type, gdf in infra_dict.items():
        style = _NCNN_STYLE.get((poi_type, infra_type), {"color": "#888888", "weight": 2})
        n_routes = int(gdf.geometry.notna().sum())
        layer_name = (
            f"NCNN: {_POI_LABEL[poi_type]} → {_INFRA_LABEL[infra_type]} "
            f"({n_routes} route{'s' if n_routes != 1 else ''})"
        )
        route_layer = folium.FeatureGroup(name=layer_name, show=True)
        for _, row in gdf.iterrows():
            if row.geometry is None:
                continue
            poi_label   = row["poi_name"]   if row["poi_name"]   else _POI_LABEL[poi_type]
            infra_label = row["infra_name"] if row["infra_name"] else _INFRA_LABEL[infra_type]
            dist_str = f"{row['route_length_m']:.0f} m" if row["route_length_m"] < float("inf") else "unreachable"
            folium.GeoJson(
                row.geometry.__geo_interface__,
                style_function=lambda _, s=style: {
                    "color":     s["color"],
                    "weight":    s["weight"],
                    "opacity":   0.85,
                    "dashArray": s.get("dashArray"),
                },
                tooltip=f"{poi_label} → {infra_label}: {dist_str}",
                popup=folium.Popup(
                    f"<b>NCNN Route</b><br>"
                    f"POI: {poi_label} ({poi_type})<br>"
                    f"Infrastructure: {infra_label} ({infra_type})<br>"
                    f"Network distance: {dist_str}",
                    max_width=320,
                ),
            ).add_to(route_layer)
        route_layer.add_to(ncnn_map)

# POI markers
for poi_type, poi_pts in [("hospital", hospitals_pts), ("fire_station", fire_stations_pts)]:
    poi_layer = folium.FeatureGroup(name=_POI_LABEL[poi_type] + " POIs", show=True)
    for _, row in poi_pts.iterrows():
        label = _get_name(row, _POI_LABEL[poi_type])
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=11, color=_POI_COLOR[poi_type], weight=2,
            fill=True, fill_color=_POI_COLOR[poi_type], fill_opacity=0.9,
            popup=folium.Popup(f"<b>{_POI_LABEL[poi_type]}</b><br>{label}", max_width=260),
            tooltip=label,
        ).add_to(poi_layer)
    poi_layer.add_to(ncnn_map)

# Infrastructure markers (representative subsets already converted to points)
for infra_type, pts_list, label in [
    ("power", [substations_pts, plants_pts],           "Power Infrastructure"),
    ("water", [water_works_pts, water_towers_pts, pumping_pts], "Water Infrastructure"),
]:
    infra_layer = folium.FeatureGroup(name=label, show=True)
    for pts in pts_list:
        for _, row in pts.iterrows():
            lbl = _get_name(row, label)
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=8, color=_INFRA_COLOR[infra_type], weight=1.5,
                fill=True, fill_color=_INFRA_COLOR[infra_type], fill_opacity=0.88,
                tooltip=lbl,
            ).add_to(infra_layer)
    infra_layer.add_to(ncnn_map)

folium.LayerControl(collapsed=False).add_to(ncnn_map)

ncnn_map_path = output_dir / "ncnn_routes_map.html"
ncnn_map.save(str(ncnn_map_path))
print(f"NCNN interactive map saved: {ncnn_map_path}")

ncnn_map